In [1]:
!pip install -q ultralytics

In [3]:

import zipfile

with zipfile.ZipFile("/content/dataset.zip", "r") as zip_ref:
    zip_ref.extractall("/content/dataset")

print("완료")


완료


In [4]:

from ultralytics import YOLO
import os

yaml_files = []

for root, dirs, files in os.walk("/content/dataset"):
    if "data.yaml" in files:
        yaml_files.append(os.path.join(root, "data.yaml"))

if not yaml_files:
    raise FileNotFoundError("오류")

data_yaml = yaml_files[0]
print("데이터셋:", data_yaml)

model = YOLO("yolo11n.pt")

results = model.train(
    data=data_yaml,
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/runs",
    name="assignment"
)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
데이터셋: /content/dataset/dataset/data.yaml
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, f

RuntimeError: Dataset '/content/dataset/dataset/data.yaml' error ❌ Dataset '/content/dataset/dataset/data.yaml' images not found, missing path '/content/dataset/images/val'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [5]:

import os
import yaml

data_yaml = "/content/dataset/dataset/data.yaml"

with open(data_yaml, "r") as f:
    data = yaml.safe_load(f)

data["path"] = "/content/dataset/dataset"

with open(data_yaml, "w") as f:
    yaml.safe_dump(data, f, allow_unicode=True)

print("완료")
print(data)


완료
{'path': '/content/dataset/dataset', 'train': 'images/train', 'val': 'images/val', 'nc': 3, 'names': {0: 'person', 1: 'bicycle', 2: 'cup'}}


In [6]:

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/dataset/dataset/data.yaml",
    epochs=30,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/runs",
    name="assignment"
)


Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=assignment-2, nbs=64, nms=None, opset=None, optimize=F

In [7]:

from google.colab import files

files.download(
    "/content/runs/assignment-2/weights/best.pt"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:

import urllib.request
import cv2

url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/solutions_ci_demo.mp4"

urllib.request.urlretrieve(url, "/content/test_video.mp4")

cap = cv2.VideoCapture("/content/test_video.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

print("영상 길이:", round(frames / fps, 2), "초")
print("FPS:", fps)

cap.release()


영상 길이: 2.07 초
FPS: 30.0


In [9]:

import cv2
from ultralytics import YOLO

model = YOLO("/content/runs/assignment-2/weights/best.pt")

cap = cv2.VideoCapture("/content/test_video.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_path = "/content/yolo_tracking_result.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    output_path, fourcc, fps, (width, height)
)

total_frames = int(fps * 5)
frame_count = 0

while frame_count < total_frames:
    ret, frame = cap.read()

    if not ret:
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        ret, frame = cap.read()
        if not ret:
            break

    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.15,
        verbose=False
    )

    annotated_frame = results[0].plot()

    out.write(annotated_frame)
    frame_count += 1

cap.release()
out.release()

print("완료")
print("저장 위치:", output_path)
print("영상 길이:", frame_count / fps, "초")


requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 2 packages in 224ms
Prepared 1 package in 64ms
Installed 1 package in 1ms
 + lap==0.5.14

requirements: AutoUpdate success ✅ 0.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

완료
저장 위치: /content/yolo_tracking_result.mp4
영상 길이: 5.0 초


In [10]:

from IPython.display import Video, display

display(Video(
    "/content/yolo_tracking_result.mp4",
    embed=True,
    width=800
))


In [11]:

!ffmpeg -y -i /content/yolo_tracking_result.mp4 -c:v libx264 -pix_fmt yuv420p -movflags +faststart /content/final_result.mp4


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [12]:

from IPython.display import Video, display

display(Video(
    "/content/final_result.mp4",
    embed=True,
    width=800
))


In [13]:

from google.colab import files

files.download("/content/final_result.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>